In [0]:
%pip install mlflow databricks-agents
dbutils.library.restartPython()

In [0]:
import mlflow

evals = [{
  "request": "Good morning",
  "response": "Good morning to you too! My email is example@example.com"
}, {
  "request": "Good afternoon, what time is it?",
  "response": "There are billions of stars in the Milky Way Galaxy."
}]

evaluation_results = mlflow.evaluate(
  data=evals,
  model_type="databricks-agent",
  # model=agent, # Uncomment to use a real model.
  evaluator_config={
    "databricks-agent": {
      # Run only this subset of built-in judges.
      "metrics": ["groundedness", "relevance_to_query", "chunk_relevance", "safety"]
    }
  }
)

# Basic Question-Answering Evaluation

In [0]:
%pip install -r ../requirements.txt

In [0]:
!pip install databricks-agent

In [0]:
dbutils.library.restartPython()

In [0]:
import openai
import pandas as pd

import mlflow

In [0]:
eval = spark.read.table("global_fs.worldbank.evaluation_dataset")
display(eval)

In [0]:
eval_pd_df = eval.selectExpr("request.messages[0].content as inputs", "expected_facts").toPandas()
display(eval_pd_df)

In [0]:
import mlflow
from mlflow.deployments import get_deploy_client

# The guidelines below will be used to evaluate any response of the agent.
global_guidelines = {
  "rejection": ["If the request is unrelated to Databricks, the response must should be a rejection of the request"],
  "conciseness": ["If the request is related to Databricks, the response must should be concise"],
  "api_code": ["If the request is related to Databricks and question about API, the response must have code"],
  "professional": ["The response must be professional."]
}

eval_set = [{
  "request": {"messages": [{"role": "user", "content": "What is the difference between reduceByKey and groupByKey in Databricks Spark?"}]}
}, {
  "request": "What is the weather today?",
}]

# Define a very simple system-prompt agent.
@mlflow.trace(span_type="AGENT")
def llama3_agent(messages):
  SYSTEM_PROMPT = """
    You are a chatbot that answers questions about Databricks.
    For requests unrelated to Databricks, reject the request.
  """
  return get_deploy_client("databricks").predict(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    inputs={"messages": [{"role": "system", "content": SYSTEM_PROMPT}, *messages]}
  )

# Evaluate the Agent with the evaluation set and log it to the MLFlow run "system_prompt_v0".
with mlflow.start_run(run_name="system_prompt_v0") as run:
  mlflow.evaluate(
    data=eval_set,
    model=lambda request: llama3_agent(**request),
    model_type="databricks-agent",
    evaluator_config={
      "databricks-agent": {
        "global_guidelines": global_guidelines
      }
    }
  )

In [0]:
eval_df = pd.DataFrame(
    {
        "inputs": [
            "How do the number and size of deposit accounts influence the share of households with bank accounts?",
            "What are the limitations in using the World Business Environment Survey data for predicting small firm banking outreach?",
            "How does the number of bank branches per 100,000 people compare between Ethiopia and Spain?",
            "What institutional factors are positively correlated with better banking system outreach?",
            "How does banking system outreach influence financing obstacles for firms?",
        ],
        "ground_truth": [
            "Both the number and size of deposit accounts influence the share of households with bank accounts: specifically, increases in the number of deposit accounts per 100,000 people and in the average deposit size are associated with increases in household bank account ownership.",
            "The World Business Environment Survey includes 120 firms per country, of which only 40% are small firms. The small sample size results in reduced data precision and predictive power.",
            "Spain has 96 bank branches per 100,000 people, while Ethiopia has fewer than one, meaning Spain has significantly more bank branches per 100,000 people than Ethiopia.",
            "Better governance and more effective credit information sharing are both positively correlated with improved banking system outreach.",
            "Banking system outreach significantly reduces financing obstacles for firms by improving access to financial services, particularly through higher branch and ATM penetration. Among these, demographic ATM penetration has the most substantial impact.",
        ],
    }
)

In [0]:
with mlflow.start_run() as run:
  system_prompt = "Answer the following question in not more than 20 words."
  basic_qa_model = mlflow.openai.log_model(
      model="gpt-4o-mini",
      task=openai.chat.completions,
      artifact_path="model",
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": "{question}"},
      ],
  )
  results = mlflow.evaluate(
      basic_qa_model.model_uri,
      eval_df,
      targets="ground_truth",  # specify which column corresponds to the expected output
      model_type="question-answering",  # model type indicates which metrics are relevant for this task
      evaluators="default",
  )
results.metrics

In [0]:
import mlflow
from mlflow.types.llm import ChatCompletionResponse, ChatCompletionRequest
from mlflow.deployments import get_deploy_client
import dataclasses

In [0]:
%pip install databricks-agents
dbutils.library.restartPython()

import mlflow
from mlflow.types.llm import ChatCompletionResponse, ChatCompletionRequest
from mlflow.deployments import get_deploy_client
import dataclasses

eval_set = [{
  "request": "What is the difference between reduceByKey and groupByKey in Databricks Spark?",
  "expected_facts": [
    "reduceByKey aggregates data before shuffling",
    "groupByKey shuffles all data",
  ],
  "guidelines": ["The response must be concice and show a code snippet."]
}, {
  "request": "What is the weather today?",
  "guidelines": ["The response must reject the request."]
}]

# Define a very simple system-prompt agent.
@mlflow.trace(span_type="AGENT")
def llama3_qna_model(messages):
  SYSTEM_PROMPT = """
    Answer the following question in not more than 20 words.
  """
  return get_deploy_client("databricks").predict(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    inputs={"messages": [{"role": "system", "content": SYSTEM_PROMPT}, *messages]}
  )

# Evaluate the agent with the evaluation set and log it to the MLFlow run "qna_eval".
with mlflow.start_run(run_name="qna_eval") as run:
  mlflow.evaluate(
    data=eval_set,
    model=lambda request: llama3_qna_model(**request),
    model_type="question-answering",  # model type indicates which metrics are relevant for this task
    evaluators="default",
  )

In [0]:
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": "{question}"},
      ],

In [0]:
def llama3_qna_model(question):
  system_prompt = "Answer the following question in not more than 20 words."
  return get_deploy_client("databricks").predict(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    inputs={"messages": [{"role": "system", "content": system_prompt}, *question]}
  )

In [0]:
llama3_qna_model("What is the difference between reduceByKey and groupByKey in Databricks Spark?")

In [0]:
with mlflow.start_run(run_name="qna_eval") as run:
    results = mlflow.evaluate(
        model=lambda request: llama3_qna_model(**request),
        data=eval_df,
        targets="expected_facts",  # specify which column corresponds to the expected output
        model_type="question-answering",  # model type indicates which metrics are relevant for this task
        evaluators="default",
    )
results.metrics

In [0]:
with mlflow.start_run(run_name="qna_eval") as run:
    results = mlflow.evaluate(
        model=lambda request: llama3_qna_model(request),
        data=eval_df,
        targets="ground_truth",
        predictions="choices",  # Specify the correct output column name here
        model_type="question-answering",
        evaluators="default",
    )
results.metrics

In [0]:
import pandas as pd

# Convert Spark DataFrame to Pandas DataFrame
eval_pd_df = eval_pd_df.toPandas()

with mlflow.start_run(run_name="qna_eval") as run:
    results = mlflow.evaluate(
        model=lambda request: llama3_qna_model(**request),
        data=eval_pd_df,
        targets="expected_facts",  # specify which column corresponds to the expected output
        model_type="question-answering",  # model type indicates which metrics are relevant for this task
        evaluators="default",
    )
results.metrics

In [0]:
from mlflow.metrics.genai import EvaluationExample, answer_similarity

# Create an example to describe what answer_similarity means like for this problem.
example = EvaluationExample(
    input="What is MLflow?",
    output="MLflow is an open-source platform for managing machine "
    "learning workflows, including experiment tracking, model packaging, "
    "versioning, and deployment, simplifying the ML lifecycle.",
    score=4,
    justification="The definition effectively explains what MLflow is "
    "its purpose, and its developer. It could be more concise for a 5-score.",
    grading_context={
        "targets": "MLflow is an open-source platform for managing "
        "the end-to-end machine learning (ML) lifecycle. It was developed by Databricks, "
        "a company that specializes in big data and machine learning solutions. MLflow is "
        "designed to address the challenges that data scientists and machine learning "
        "engineers face when developing, training, and deploying machine learning models."
    },
)

# Construct the metric using OpenAI GPT-4 as the judge
answer_similarity_metric = answer_similarity(model="openai:/gpt-4", examples=[example])

print(answer_similarity_metric)

In [0]:
with mlflow.start_run() as run:
    results = mlflow.evaluate(
        basic_qa_model.model_uri,
        eval_df,
        targets="ground_truth",
        model_type="question-answering",
        evaluators="default",
        extra_metrics=[answer_similarity_metric],  # use the answer similarity metric created above
    )
results.metrics

In [0]:
results.tables["eval_results_table"]

In [0]:
from mlflow.metrics.genai import EvaluationExample, make_genai_metric

professionalism_metric = make_genai_metric(
    name="professionalism",
    definition=(
        "Professionalism refers to the use of a formal, respectful, and appropriate style of communication that is tailored to the context and audience. It often involves avoiding overly casual language, slang, or colloquialisms, and instead using clear, concise, and respectful language"
    ),
    grading_prompt=(
        "Professionalism: If the answer is written using a professional tone, below "
        "are the details for different scores: "
        "- Score 1: Language is extremely casual, informal, and may include slang or colloquialisms. Not suitable for professional contexts."
        "- Score 2: Language is casual but generally respectful and avoids strong informality or slang. Acceptable in some informal professional settings."
        "- Score 3: Language is balanced and avoids extreme informality or formality. Suitable for most professional contexts. "
        "- Score 4: Language is noticeably formal, respectful, and avoids casual elements. Appropriate for business or academic settings. "
        "- Score 5: Language is excessively formal, respectful, and avoids casual elements. Appropriate for the most formal settings such as textbooks. "
    ),
    examples=[
        EvaluationExample(
            input="What is MLflow?",
            output=(
                "MLflow is like your friendly neighborhood toolkit for managing your machine learning projects. It helps you track experiments, package your code and models, and collaborate with your team, making the whole ML workflow smoother. It's like your Swiss Army knife for machine learning!"
            ),
            score=2,
            justification=(
                "The response is written in a casual tone. It uses contractions, filler words such as 'like', and exclamation points, which make it sound less professional. "
            ),
        )
    ],
    version="v1",
    model="openai:/gpt-4",
    parameters={"temperature": 0.0},
    grading_context_columns=[],
    aggregations=["mean", "variance", "p90"],
    greater_is_better=True,
)

print(professionalism_metric)

In [0]:
with mlflow.start_run() as run:
    results = mlflow.evaluate(
        basic_qa_model.model_uri,
        eval_df,
        model_type="question-answering",
        evaluators="default",
        extra_metrics=[professionalism_metric],  # use the professionalism metric we created above
    )
print(results.metrics)

In [0]:
results.tables["eval_results_table"]

In [0]:
with mlflow.start_run() as run:
    system_prompt = "Answer the following question using extreme formality."
    professional_qa_model = mlflow.openai.log_model(
        model="gpt-4o-mini",
        task=openai.chat.completions,
        artifact_path="model",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": "{question}"},
        ],
    )
    results = mlflow.evaluate(
        professional_qa_model.model_uri,
        eval_df,
        model_type="question-answering",
        evaluators="default",
        extra_metrics=[professionalism_metric],
    )
print(results.metrics)

In [0]:
results.tables["eval_results_table"]